In [98]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress
import glob
import pandas as pd
import scipy
import xrft as xt

In [99]:
ds1 = xr.open_dataset('Hplus_ssp585_biascorrected.nc')
ds2 = xr.open_dataset('Hplus_ssp126_biascorrected.nc')
ds3 = xr.open_dataset('Hplus_ssp245_biascorrected.nc')


In [ ]:
lat_range = slice(-30, 30)
lon_range = slice(30, 120)
month_hist  = ds2.sel(lat=lat_range, lon=lon_range, time=slice('1985', '2014'))
month_126  = ds2.sel(lat=lat_range, lon=lon_range, time=slice('2071', '2100'))
month_245  = ds3.sel(lat=lat_range, lon=lon_range, time=slice('2071', '2100'))
month_585  = ds1.sel(lat=lat_range, lon=lon_range, time=slice('2071', '2100'))

# print(month_hist)
# print(month_126)
# print(month_245)
# print(month_585)

In [ ]:

# Select 2015–2100
ph = month_126['ph'].sel(time=slice('2015-01-01', '2100-12-31'))

# Convert time to decimal years
time_years = (
    ph['time'] - ph['time'].isel(time=0)
) / np.timedelta64(1, 'D') / 365.25


# Function to calculate slope and p-value
def calculate_trend(y, x):
    mask = np.isfinite(y) & np.isfinite(x)

    if np.sum(mask) < 3:
        return np.nan, np.nan

    result = linregress(x[mask], y[mask])

    return result.slope, result.pvalue


# Calculate trend and p-value
trend, pvalue = xr.apply_ufunc(
    calculate_trend,
    ph,
    time_years,
    input_core_dims=[['time'], ['time']],
    output_core_dims=[[], []],
    vectorize=True,
    dask='parallelized',
    output_dtypes=[float, float]
)

# Assign names
trend = trend.rename('ph_trend')
pvalue = pvalue.rename('pvalue')

# print(trend)
# print(pvalue)

In [103]:
import xarray as xr
import numpy as np
from scipy.stats import linregress

# Select 2015–2100
ph = month_245['ph'].sel(time=slice('2015-01-01', '2100-12-31'))

# Convert time to decimal years
time_years = (
    ph['time'] - ph['time'].isel(time=0)
) / np.timedelta64(1, 'D') / 365.25


# Function to calculate slope and p-value
def calculate_trend(y, x):
    mask = np.isfinite(y) & np.isfinite(x)

    if np.sum(mask) < 3:
        return np.nan, np.nan

    result = linregress(x[mask], y[mask])

    return result.slope, result.pvalue


# Calculate trend and p-value
trend_245, pvalue_245 = xr.apply_ufunc(
    calculate_trend,
    ph,
    time_years,
    input_core_dims=[['time'], ['time']],
    output_core_dims=[[], []],
    vectorize=True,
    dask='parallelized',
    output_dtypes=[float, float]
)

# Assign names
trend_245 = trend_245.rename('ph_trend')
pvalue_245 = pvalue_245.rename('pvalue')

# print(trend)
# print(pvalue)

In [ ]:
#

In [106]:


def calculate_spatial_statistics(ds):

    # Extract pH
    ph = ds['ph']

    time_years = np.arange(ph.sizes['time']) / 12.0

    def trend_pvalue(y, x):

        mask = np.isfinite(y)

        if np.sum(mask) < 3:
            return np.nan, np.nan

        result = linregress(x[mask], y[mask])

        return result.slope, result.pvalue

    trend, pvalue = xr.apply_ufunc(
        trend_pvalue,
        ph,
        xr.DataArray(time_years, dims='time'),
        input_core_dims=[['time'], ['time']],
        output_core_dims=[[], []],
        vectorize=True,
        dask='parallelized',
        output_dtypes=[float, float]
    )

    # Spatial standard deviation

    std = ph.std(dim='time', skipna=True)

    trend.name = 'ph_trend'
    pvalue.name = 'pvalue'
    std.name = 'ph_std'

    # Add units
    trend.attrs['units'] = 'pH units yr-1'
    pvalue.attrs['description'] = 'p-value of linear pH trend'
    std.attrs['units'] = 'pH units'

    return trend, pvalue, std

In [ ]:
hist_trend, hist_pvalue, hist_std = calculate_spatial_statistics(month_hist)

ssp126_trend, ssp126_pvalue, ssp126_std = calculate_spatial_statistics(month_126)

ssp245_trend, ssp245_pvalue, ssp245_std = calculate_spatial_statistics(month_245)

ssp585_trend, ssp585_pvalue, ssp585_std = calculate_spatial_statistics(month_585)

In [110]:
hist_trend.to_netcdf(
    'pH_trend_hist.nc'
)

hist_pvalue.to_netcdf(
    'pH_pvalue_hist.nc'
)

hist_std.to_netcdf(
    'pH_std_hist.nc'
)

In [111]:
ssp126_trend.to_netcdf(
    'pH_trend_SSP126_FF.nc'
)

ssp126_pvalue.to_netcdf(
    'pH_pvalue_SSP126_FF.nc'
)

ssp126_std.to_netcdf(
    'pH_std_SSP126_FF.nc'
)

In [112]:
ssp245_trend.to_netcdf(
    'pH_trend_SSP245_FF.nc'
)

ssp245_pvalue.to_netcdf(
    'pH_pvalue_SSP245_FF.nc'
)

ssp245_std.to_netcdf(
    'pH_std_SSP245_FF.nc'
)

In [113]:
ssp585_trend.to_netcdf(
    'pH_trend_SSP585_FF.nc'
)

ssp585_pvalue.to_netcdf(
    'pH_pvalue_SSP585_FF.nc'
)

ssp126_std.to_netcdf(
    'pH_std_SSP585_FF.nc'
)

# Plot

In [ ]:
trend_hist = xr.open_dataset('pH_trend_hist.nc')
trend_126 = xr.open_dataset('pH_trend_SSP126_FF.nc')
trend_245 = xr.open_dataset('pH_trend_SSP245_FF.nc')
trend_585 = xr.open_dataset('pH_trend_SSP585_FF.nc')

std_hist = xr.open_dataset('pH_std_hist.nc')
std_126 = xr.open_dataset('pH_std_SSP126_FF.nc')
std_245 = xr.open_dataset('pH_std_SSP245_FF.nc')
std_585 = xr.open_dataset('pH_std_SSP585_FF.nc')


toe_hist = xr.open_dataset('Toe_hist_bc.nc')
toe_126 = xr.open_dataset('Toe_126_bc.nc')
toe_245 = xr.open_dataset('Toe_245_bc.nc')
toe_585 = xr.open_dataset('Toe_585_bc.nc')


trend_hist = (trend_hist*(10**9))
trend_126 = (trend_126*(10**9))
trend_245 = (trend_245*(10**9))
trend_585 = (trend_585*(10**9))

std_hist = (std_hist*(10**9))
std_126 = (std_126*(10**9))
std_245 = (std_245*(10**9))
std_585 = (std_585*(10**9))

In [ ]:
from matplotlib import colors
from mpl_toolkits.basemap import Basemap


trend_data = [
    trend_hist['ph_trend'],
    trend_585['ph_trend'],
    trend_245['ph_trend'],
    trend_126['ph_trend']
]

titles = [
    'Historical',
    'SSP5-8.5',
    'SSP2-4.5',
    'SSP1-2.6'
]

subplot_labels = [
    '(a)', '(b)', '(c)', '(d)'
]


plt.rcParams['font.weight'] = 'bold'
plt.rcParams['axes.labelweight'] = 'bold'
plt.rcParams['xtick.labelsize'] = 13
plt.rcParams['ytick.labelsize'] = 13



levels_trend = np.arange(
    -0.1,
    0.101,
    0.01
)

norm_trend = colors.BoundaryNorm(
    levels_trend,
    ncolors=256
)


fig, axes = plt.subplots(
    nrows=2,
    ncols=2,
    figsize=(13, 9)
)

axes = axes.flatten()

contours = []


for i, (ax, data, title, label) in enumerate(
        zip(
            axes,
            trend_data,
            titles,
            subplot_labels
        )):

 
    m = Basemap(
        llcrnrlat=-30.1,
        urcrnrlat=30.1,
        llcrnrlon=30,
        urcrnrlon=120.1,
        resolution='c',
        ax=ax
    )



    m.fillcontinents(
        color='gray',
        lake_color='white',
        zorder=2
    )

    m.drawcoastlines(
        linewidth=1.2,
        zorder=3
    )




    if i in [0, 2]:

        m.drawparallels(
            range(-90, 120, 10),
            labels=[1, 0, 0, 1],
            linewidth=0.001,
            dashes=[4, 2],
            color='w',
            fontsize=11
        )

    else:

        m.drawparallels(
            range(-90, 120, 10),
            labels=[0, 0, 0, 0],
            linewidth=0.001,
            dashes=[4, 2],
            color='w'
        )




    if i in [2, 3]:

        m.drawmeridians(
            range(30, 121, 30),
            labels=[1, 0, 0, 1],
            linewidth=0.001,
            color='w',
            fontsize=11
        )

    else:

        m.drawmeridians(
            range(30, 121, 30),
            labels=[0, 0, 0, 0],
            linewidth=0.001,
            color='w'
        )


    lon, lat = np.meshgrid(
        data.lon.values,
        data.lat.values
    )

    x, y = m(
        lon,
        lat
    )



    cf = m.contourf(
        x,
        y,
        data.values,
        levels=levels_trend,
        norm=norm_trend,
        cmap='RdBu_r',
        extend='both',
        zorder=1
    )

    contours.append(cf)


    ax.text(
        0.89,
        0.96,
        label,
        transform=ax.transAxes,
        fontsize=15,
        fontweight='bold',
        va='top',
        ha='left',
        color='white'
    )




    ax.set_title(
        title,
        fontsize=14,
        fontweight='bold'
    )


cbar_ax = fig.add_axes(
    [0.20, 0.06, 0.60, 0.025]
)

cbar = fig.colorbar(
    contours[0],
    cax=cbar_ax,
    orientation='horizontal',
    ticks=[
        -0.1,
        -0.05,
         0.0,
         0.05,
         0.1
    ],
    extend='both'
)

cbar.ax.tick_params(
    axis='x',
    which='major',
    labelsize=12,
    width=1.8,
    length=5
)

for tick in cbar.ax.get_xticklabels():
    tick.set_fontweight('bold')


cbar.set_label(
    r'[H⁺] trend (nmol kg$^{-1}$ yr$^{-1}$)',
    fontsize=13,
    fontweight='bold'
)

for ax in axes:

    for tick in ax.get_xticklabels():
        tick.set_fontweight('bold')

    for tick in ax.get_yticklabels():
        tick.set_fontweight('bold')


plt.subplots_adjust(
    left=0.08,
    right=0.96,
    bottom=0.14,
    top=0.91,
    wspace=-0.13,
    hspace=0.13
)

# plt.savefig(
#     'pH_trend_2x2_blue_red.png',
#     dpi=300,
#     bbox_inches='tight'
# )

plt.show()